# DAG Assumption Tests for Latent Variable SEM (v4)

This notebook tests conditional independence assumptions to justify the SEM structure.

**Model Structure (v4):**
- Latent `leisure_participation` indicated by `hill_q1` (diversity) and `mean_leisure_duration`
- STA (`ak_ihs`) → Travel Time → Leisure Participation
- Covariates: sociodemographics + transport mode

**Tests:**
1. Do diversity and duration measure the same construct? (correlation, factor analysis)
2. Does Individual → Travel Time path exist? (conditional on STA)
3. Does Transport Mode → Leisure Participation path exist? (conditional on STA, Individual)
4. Does Transport Mode → Travel Time path exist?
5. Does Individual → STA path exist? (conditional on Transport Mode)
6. Is Travel Time a mediator? (Sobel test / mediation analysis)

In [1]:
%load_ext autoreload
%autoreload 2
%cd D:\netmob25

D:\netmob25


In [2]:
import os
os.environ['USE_PYGEOS'] = '0'
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
from factor_analyzer import FactorAnalyzer, calculate_kmo, calculate_bartlett_sphericity
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv("dbs/data_p/commuter_model_features_r.csv")
print(f"Loaded {len(df)} rows")
print(f"\nColumns: {list(df.columns)}")

Loaded 2444 rows

Columns: ['ID', 'entropy_mm', 'activity_nh_ratio', 'activity_time_third', 'total_travel_time', 'xs_total_hws', 'trip_chaining_presence', 'hill_q1', 'mean_leisure_duration', 'Gender', 'Age', 'Education', 'Household_type', 'weight_ind', 'main_mode_r', 'access_h', 'mode', 'ak', 'ak_cat', 'ak_log', 'ak_ihs', 'ak_log1p', 'codgeo', 'pt_sub', 'active_mode', 'poverty_rate']


In [4]:
# Prepare variables
# Use ak_90_ihs as the baseline STA (or ak_ihs if already created)
if 'ak_90_ihs' in df.columns:
    df['sta'] = df['ak_90_ihs']
elif 'ak_ihs' in df.columns:
    df['sta'] = df['ak_ihs']
else:
    df['sta'] = np.arcsinh(df['ak'])

# Binary variables
df['female'] = (df['Gender'] == 'Woman').astype(int)
df['edu_high'] = df['Education'].isin([5, 9]).astype(int)
df['mode_pt'] = (df['mode'] == 'Public transit').astype(int)
df['hh_couple_child'] = (df['Household_type'] == 6).astype(int)

# Ensure numeric
df['active_mode'] = pd.to_numeric(df['active_mode'], errors='coerce').fillna(0)
df['pt_sub'] = df['pt_sub'].apply(lambda x: 1 if str(x).lower() in ['true', '1', 'yes'] else 0)

# Standardize continuous variables for comparison
for col in ['sta', 'total_travel_time', 'hill_q1', 'mean_leisure_duration', 'poverty_rate']:
    if col in df.columns:
        df[f'{col}_z'] = (df[col] - df[col].mean()) / df[col].std()

print("Variables prepared.")
print(f"STA range: {df['sta'].min():.2f} to {df['sta'].max():.2f}")

Variables prepared.
STA range: 0.00 to 11.25


## Test 1: Do Diversity and Duration Measure the Same Construct?

If both are valid indicators of "leisure participation", they should:
- Be positively correlated
- Load on a single factor
- Pass KMO and Bartlett's test for factor analysis

In [12]:
# Correlation between the two indicators
corr = df[['hill_q1', 'mean_leisure_duration']].corr()
print("=== Correlation Matrix ===")
print(corr.round(3))

# Weighted correlation
def weighted_corr(x, y, w):
    """Compute weighted Pearson correlation"""
    mask = ~(np.isnan(x) | np.isnan(y) | np.isnan(w))
    x, y, w = x[mask], y[mask], w[mask]
    w_sum = w.sum()
    mean_x = np.average(x, weights=w)
    mean_y = np.average(y, weights=w)
    cov_xy = np.sum(w * (x - mean_x) * (y - mean_y)) / w_sum
    std_x = np.sqrt(np.sum(w * (x - mean_x)**2) / w_sum)
    std_y = np.sqrt(np.sum(w * (y - mean_y)**2) / w_sum)
    return cov_xy / (std_x * std_y)

wcorr = weighted_corr(
    df['hill_q1'].values, 
    df['mean_leisure_duration'].values,
    df['weight_ind'].values
)
print(f"\nWeighted correlation: {wcorr:.3f}")

=== Correlation Matrix ===
                       hill_q1  mean_leisure_duration
hill_q1                  1.000                  0.286
mean_leisure_duration    0.286                  1.000

Weighted correlation: 0.309


In [13]:
# Factor analysis
indicators = df[['hill_q1', 'mean_leisure_duration']].dropna()

# KMO test (should be > 0.5)
try:
    kmo_all, kmo_model = calculate_kmo(indicators)
    print(f"KMO measure of sampling adequacy: {kmo_model:.3f}")
    print(f"  (>0.5 acceptable, >0.7 good, >0.8 excellent)")
except:
    print("KMO calculation failed (likely due to 2 variables only)")

# Bartlett's test (should be significant)
chi2, p = calculate_bartlett_sphericity(indicators)
print(f"\nBartlett's test: chi2={chi2:.1f}, p={p:.2e}")
print(f"  (p < 0.05 suggests variables are correlated, factor analysis appropriate)")

KMO measure of sampling adequacy: 0.500
  (>0.5 acceptable, >0.7 good, >0.8 excellent)

Bartlett's test: chi2=208.7, p=2.66e-47
  (p < 0.05 suggests variables are correlated, factor analysis appropriate)


In [14]:
# Single factor extraction
fa = FactorAnalyzer(n_factors=1, rotation=None)
fa.fit(indicators)

print("=== Single Factor Loadings ===")
loadings = pd.DataFrame(
    fa.loadings_, 
    index=['hill_q1', 'mean_leisure_duration'],
    columns=['Factor1']
)
print(loadings.round(3))

# Variance explained
var_explained = fa.get_factor_variance()
print(f"\nVariance explained: {var_explained[1][0]*100:.1f}%")

print("\n=== Conclusion ===")
# Check ABSOLUTE values of loadings (sign is arbitrary in factor analysis)
min_abs_loading = loadings['Factor1'].abs().min()
print(f"Minimum absolute loading: {min_abs_loading:.3f}")

if min_abs_loading > 0.4:
    print(f"|{min_abs_loading:.3f}| > 0.4: Both indicators load adequately on a single factor.")
    print("Latent variable approach is JUSTIFIED.")
else:
    print(f"|{min_abs_loading:.3f}| <= 0.4: WARNING - Weak loadings.")
    print("Consider separate outcome models.")

=== Single Factor Loadings ===
                       Factor1
hill_q1                 -0.535
mean_leisure_duration   -0.535

Variance explained: 28.6%

=== Conclusion ===
Minimum absolute loading: 0.535
|0.535| > 0.4: Both indicators load adequately on a single factor.
Latent variable approach is JUSTIFIED.


## Test 2: Individual → Travel Time | STA

Test if sociodemographic variables predict travel time beyond what STA explains.

H0: Individual ⊥ Travel Time | STA (no path needed)
H1: Path from Individual to Travel Time exists

In [15]:
# Model 1: Travel Time ~ STA only
m1 = smf.wls('total_travel_time ~ sta', data=df, weights=df['weight_ind']).fit()

# Model 2: Travel Time ~ STA + Individual (sociodemographics)
m2 = smf.wls('total_travel_time ~ sta + female + edu_high + poverty_rate + hh_couple_child', 
             data=df, weights=df['weight_ind']).fit()

# Compare nested models
anova_result = sm.stats.anova_lm(m1, m2)
print("=== Test: Individual → Travel Time | STA ===")
print(anova_result)

p_value = anova_result['Pr(>F)'].iloc[1]
print(f"\n=== Conclusion ===")
if p_value < 0.05:
    print(f"p = {p_value:.4f} < 0.05: REJECT H0")
    print("Individual → Travel Time path is NEEDED.")
else:
    print(f"p = {p_value:.4f} >= 0.05: FAIL TO REJECT H0")
    print("Individual → Travel Time path may be omitted.")

=== Test: Individual → Travel Time | STA ===
   df_resid           ssr  df_diff       ss_diff          F        Pr(>F)
0    2442.0  1.046138e+10      0.0           NaN        NaN           NaN
1    2438.0  1.024785e+10      4.0  2.135258e+08  12.699632  3.130153e-10

=== Conclusion ===
p = 0.0000 < 0.05: REJECT H0
Individual → Travel Time path is NEEDED.


## Test 3: Transport Mode → Travel Time | STA, Individual

Test if transport mode variables predict travel time beyond STA and demographics.

In [16]:
# Model 1: Travel Time ~ STA + Individual
m1 = smf.wls('total_travel_time ~ sta + female + edu_high + poverty_rate + hh_couple_child', 
             data=df, weights=df['weight_ind']).fit()

# Model 2: + Transport Mode
m2 = smf.wls('total_travel_time ~ sta + female + edu_high + poverty_rate + hh_couple_child + '
             'mode_pt + pt_sub + active_mode', 
             data=df, weights=df['weight_ind']).fit()

anova_result = sm.stats.anova_lm(m1, m2)
print("=== Test: Transport Mode → Travel Time | STA, Individual ===")
print(anova_result)

p_value = anova_result['Pr(>F)'].iloc[1]
print(f"\n=== Conclusion ===")
if p_value < 0.05:
    print(f"p = {p_value:.4f} < 0.05: REJECT H0")
    print("Transport Mode → Travel Time path is NEEDED.")
else:
    print(f"p = {p_value:.4f} >= 0.05: FAIL TO REJECT H0")
    print("Transport Mode → Travel Time path may be omitted.")

=== Test: Transport Mode → Travel Time | STA, Individual ===
   df_resid           ssr  df_diff       ss_diff          F        Pr(>F)
0    2438.0  1.024785e+10      0.0           NaN        NaN           NaN
1    2435.0  9.950140e+09      3.0  2.977138e+08  24.285526  1.754700e-15

=== Conclusion ===
p = 0.0000 < 0.05: REJECT H0
Transport Mode → Travel Time path is NEEDED.


## Test 4: Transport Mode → Leisure Participation | STA, Travel Time, Individual

Test if transport mode has direct effect on leisure participation beyond STA, travel time, and demographics.

In [17]:
# Test for diversity (hill_q1)
m1_div = smf.wls('hill_q1 ~ sta + total_travel_time + female + edu_high + poverty_rate + hh_couple_child', 
                 data=df, weights=df['weight_ind']).fit()

m2_div = smf.wls('hill_q1 ~ sta + total_travel_time + female + edu_high + poverty_rate + hh_couple_child + '
                 'mode_pt + pt_sub + active_mode', 
                 data=df, weights=df['weight_ind']).fit()

anova_div = sm.stats.anova_lm(m1_div, m2_div)
print("=== Test: Transport Mode → Diversity | STA, TT, Individual ===")
print(anova_div)

# Test for duration
m1_dur = smf.wls('mean_leisure_duration ~ sta + total_travel_time + female + edu_high + poverty_rate + hh_couple_child', 
                 data=df, weights=df['weight_ind']).fit()

m2_dur = smf.wls('mean_leisure_duration ~ sta + total_travel_time + female + edu_high + poverty_rate + hh_couple_child + '
                 'mode_pt + pt_sub + active_mode', 
                 data=df, weights=df['weight_ind']).fit()

anova_dur = sm.stats.anova_lm(m1_dur, m2_dur)
print("\n=== Test: Transport Mode → Duration | STA, TT, Individual ===")
print(anova_dur)

print("\n=== Conclusion ===")
p_div = anova_div['Pr(>F)'].iloc[1]
p_dur = anova_dur['Pr(>F)'].iloc[1]
print(f"Diversity: p = {p_div:.4f} {'< 0.05 (NEEDED)' if p_div < 0.05 else '>= 0.05 (may omit)'}")
print(f"Duration:  p = {p_dur:.4f} {'< 0.05 (NEEDED)' if p_dur < 0.05 else '>= 0.05 (may omit)'}")

=== Test: Transport Mode → Diversity | STA, TT, Individual ===
   df_resid           ssr  df_diff        ss_diff      F        Pr(>F)
0    2437.0  1.149764e+07      0.0            NaN    NaN           NaN
1    2434.0  1.114502e+07      3.0  352620.212291  25.67  2.402686e-16

=== Test: Transport Mode → Duration | STA, TT, Individual ===
   df_resid           ssr  df_diff       ss_diff         F    Pr(>F)
0    2437.0  5.748410e+10      0.0           NaN       NaN       NaN
1    2434.0  5.695930e+10      3.0  5.248031e+08  7.475342  0.000056

=== Conclusion ===
Diversity: p = 0.0000 < 0.05 (NEEDED)
Duration:  p = 0.0001 < 0.05 (NEEDED)


## Test 5: Individual → STA | Transport Mode

Test if sociodemographics predict STA beyond transport mode.

In [18]:
# Model 1: STA ~ Transport Mode only
m1 = smf.wls('sta ~ mode_pt + pt_sub + active_mode', 
             data=df, weights=df['weight_ind']).fit()

# Model 2: STA ~ Transport Mode + Individual
m2 = smf.wls('sta ~ mode_pt + pt_sub + active_mode + female + edu_high + poverty_rate + hh_couple_child', 
             data=df, weights=df['weight_ind']).fit()

anova_result = sm.stats.anova_lm(m1, m2)
print("=== Test: Individual → STA | Transport Mode ===")
print(anova_result)

p_value = anova_result['Pr(>F)'].iloc[1]
print(f"\n=== Conclusion ===")
if p_value < 0.05:
    print(f"p = {p_value:.4f} < 0.05: REJECT H0")
    print("Individual → STA path is NEEDED.")
else:
    print(f"p = {p_value:.4f} >= 0.05: FAIL TO REJECT H0")
    print("Individual → STA path may be omitted.")

=== Test: Individual → STA | Transport Mode ===
   df_resid           ssr  df_diff        ss_diff       F    Pr(>F)
0    2440.0  9.564601e+07      0.0            NaN     NaN       NaN
1    2436.0  9.496645e+07      4.0  679563.698703  4.3579  0.001633

=== Conclusion ===
p = 0.0016 < 0.05: REJECT H0
Individual → STA path is NEEDED.


## Test 6: Mediation Analysis (STA → Travel Time → Leisure Participation)

Test if travel time mediates the effect of STA on leisure participation.

In [19]:
# Create a simple leisure participation proxy (average of standardized indicators)
df['leisure_part_proxy'] = (df['hill_q1_z'] + df['mean_leisure_duration_z']) / 2

# Step 1: Total effect (c path)
total = smf.wls('leisure_part_proxy ~ sta + female + edu_high + poverty_rate + hh_couple_child + '
                'mode_pt + pt_sub + active_mode', 
                data=df, weights=df['weight_ind']).fit()
c_total = total.params['sta']
c_total_p = total.pvalues['sta']

# Step 2: a path (STA -> Travel Time)
a_model = smf.wls('total_travel_time ~ sta + female + edu_high + poverty_rate + hh_couple_child + '
                  'mode_pt + pt_sub + active_mode', 
                  data=df, weights=df['weight_ind']).fit()
a = a_model.params['sta']
a_se = a_model.bse['sta']

# Step 3: b path (Travel Time -> Leisure, controlling for STA)
b_model = smf.wls('leisure_part_proxy ~ sta + total_travel_time + female + edu_high + poverty_rate + '
                  'hh_couple_child + mode_pt + pt_sub + active_mode', 
                  data=df, weights=df['weight_ind']).fit()
b = b_model.params['total_travel_time']
b_se = b_model.bse['total_travel_time']
c_prime = b_model.params['sta']  # Direct effect
c_prime_p = b_model.pvalues['sta']

# Indirect effect
indirect = a * b

# Sobel test
sobel_se = np.sqrt(a**2 * b_se**2 + b**2 * a_se**2)
sobel_z = indirect / sobel_se
sobel_p = 2 * (1 - stats.norm.cdf(abs(sobel_z)))

print("=== Mediation Analysis ===")
print(f"\nPath coefficients:")
print(f"  a (STA → Travel Time):        {a:.4f}")
print(f"  b (Travel Time → Leisure):    {b:.4f}")
print(f"  c' (Direct: STA → Leisure):   {c_prime:.4f} (p={c_prime_p:.4f})")
print(f"  c (Total: STA → Leisure):     {c_total:.4f} (p={c_total_p:.4f})")

print(f"\nIndirect effect (a × b): {indirect:.4f}")
print(f"Sobel test: z = {sobel_z:.3f}, p = {sobel_p:.4f}")

print(f"\n=== Conclusion ===")
if sobel_p < 0.05:
    print(f"Sobel p = {sobel_p:.4f} < 0.05: Mediation is SIGNIFICANT.")
    print("Travel Time mediates the STA → Leisure relationship.")
    if c_prime_p < 0.05:
        print("Direct effect also significant → PARTIAL mediation.")
    else:
        print("Direct effect not significant → FULL mediation.")
else:
    print(f"Sobel p = {sobel_p:.4f} >= 0.05: Mediation is NOT significant.")

=== Mediation Analysis ===

Path coefficients:
  a (STA → Travel Time):        -3.5199
  b (Travel Time → Leisure):    0.0017
  c' (Direct: STA → Leisure):   0.0212 (p=0.0000)
  c (Total: STA → Leisure):     0.0153 (p=0.0001)

Indirect effect (a × b): -0.0059
Sobel test: z = -4.377, p = 0.0000

=== Conclusion ===
Sobel p = 0.0000 < 0.05: Mediation is SIGNIFICANT.
Travel Time mediates the STA → Leisure relationship.
Direct effect also significant → PARTIAL mediation.


## Test 7: Which Household Type Dummies Are Needed?

Test individual household type effects to see which should be retained.

In [20]:
# Full model with all household types
model_full = smf.wls('leisure_part_proxy ~ sta + total_travel_time + female + edu_high + poverty_rate + '
                     'C(Household_type) + mode_pt + pt_sub + active_mode', 
                     data=df, weights=df['weight_ind']).fit()

print("=== Household Type Effects on Leisure Participation ===")
hh_params = [(k, v, model_full.pvalues[k]) 
             for k, v in model_full.params.items() 
             if 'Household_type' in k]

print(f"{'Household Type':<30} {'Coef':>10} {'p-value':>10} {'Sig':>5}")
print("-" * 60)
for name, coef, pval in hh_params:
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    print(f"{name:<30} {coef:>10.4f} {pval:>10.4f} {sig:>5}")

print("\n=== Conclusion ===")
sig_hh = [name for name, coef, pval in hh_params if pval < 0.05]
if sig_hh:
    print(f"Significant household types: {sig_hh}")
    print("Include only these in the SEM.")
else:
    print("No household types are significant. May omit from model.")

=== Household Type Effects on Leisure Participation ===
Household Type                       Coef    p-value   Sig
------------------------------------------------------------
C(Household_type)[T.1.0]          -0.0325     0.5522      
C(Household_type)[T.2.0]          -0.2068     0.0010   ***
C(Household_type)[T.3.0]          -0.0314     0.6730      
C(Household_type)[T.4.0]           0.3994     0.0073    **
C(Household_type)[T.5.0]           0.0261     0.9238      
C(Household_type)[T.6.0]          -0.3223     0.0000   ***
C(Household_type)[T.7.0]           0.0938     0.5271      

=== Conclusion ===
Significant household types: ['C(Household_type)[T.2.0]', 'C(Household_type)[T.4.0]', 'C(Household_type)[T.6.0]']
Include only these in the SEM.


## Summary of DAG Tests

## Test 8: Individual → Transport Mode

Test if sociodemographic variables predict transport mode choices.

This tests whether paths from Individual attributes to Transport Mode should exist in the DAG.
Even if significant, we may choose to treat transport mode as exogenous for practical reasons
(mode choice is complex, and including these paths caused model non-convergence in v5).

In [6]:
# Test Individual → Transport Mode for each mode variable
print("=== Test 8: Individual → Transport Mode ===\n")

mode_vars = ['mode_pt', 'active_mode', 'pt_sub']
socio_predictors = 'female + edu_high + poverty_rate + hh_couple_child'

results = []
for mode_var in mode_vars:
    # Null model (intercept only)
    m0 = smf.wls(f'{mode_var} ~ 1', data=df, weights=df['weight_ind']).fit()
    
    # Full model with sociodemographics
    m1 = smf.wls(f'{mode_var} ~ {socio_predictors}', data=df, weights=df['weight_ind']).fit()
    
    # ANOVA comparison
    anova_result = sm.stats.anova_lm(m0, m1)
    f_stat = anova_result['F'].iloc[1]
    p_value = anova_result['Pr(>F)'].iloc[1]
    r2 = m1.rsquared
    
    results.append({
        'outcome': mode_var,
        'F': f_stat,
        'p': p_value,
        'R2': r2,
        'significant': p_value < 0.05
    })
    
    print(f"--- {mode_var} ~ Individual ---")
    print(f"F = {f_stat:.2f}, p = {p_value:.4f}, R² = {r2:.3f}")
    
    # Show individual coefficients
    print("Coefficients:")
    for var in ['female', 'edu_high', 'poverty_rate', 'hh_couple_child']:
        coef = m1.params[var]
        pval = m1.pvalues[var]
        sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
        print(f"  {var}: {coef:.4f} (p={pval:.4f}) {sig}")
    print()

print("=== Conclusion ===")
for r in results:
    status = "SIGNIFICANT - path exists" if r['significant'] else "Not significant - path may be omitted"
    print(f"{r['outcome']}: p={r['p']:.4f} → {status}")

=== Test 8: Individual → Transport Mode ===

--- mode_pt ~ Individual ---
F = 41.62, p = 0.0000, R² = 0.064
Coefficients:
  female: -0.0304 (p=0.1129) 
  edu_high: 0.1632 (p=0.0000) ***
  poverty_rate: 0.0064 (p=0.0000) ***
  hh_couple_child: -0.1602 (p=0.0000) ***

--- active_mode ~ Individual ---
F = 24.02, p = 0.0000, R² = 0.038
Coefficients:
  female: -0.0667 (p=0.0006) ***
  edu_high: 0.1808 (p=0.0000) ***
  poverty_rate: 0.0012 (p=0.3070) 
  hh_couple_child: 0.0131 (p=0.5008) 

--- pt_sub ~ Individual ---
F = 32.58, p = 0.0000, R² = 0.051
Coefficients:
  female: 0.0009 (p=0.9615) 
  edu_high: 0.0993 (p=0.0000) ***
  poverty_rate: 0.0056 (p=0.0000) ***
  hh_couple_child: -0.1704 (p=0.0000) ***

=== Conclusion ===
mode_pt: p=0.0000 → SIGNIFICANT - path exists
active_mode: p=0.0000 → SIGNIFICANT - path exists
pt_sub: p=0.0000 → SIGNIFICANT - path exists


In [7]:
print("="*60)
print("SUMMARY: DAG Path Justification for SEM v4")
print("="*60)
print("""
Test 1: Latent Variable Justification
  - Diversity and duration are correlated and load on single factor
  - Latent 'leisure_participation' is justified

Test 2-5: Path Necessity Tests
  - Use ANOVA nested model comparisons
  - p < 0.05 → path is needed
  - p >= 0.05 → path may be omitted for parsimony

Test 6: Mediation Test
  - Sobel test for indirect effect via travel time
  - Significant mediation supports the hypothesized mechanism

Test 7: Covariate Selection
  - Only include significant household type dummies

Test 8: Individual → Transport Mode
  - Tests whether sociodemographics predict mode choice
  - Even if significant, paths are OMITTED because:
    a) Including them caused model non-convergence
    b) Mode choice is complex, better modeled separately
    c) Transport mode treated as exogenous in our framework
""")

print("\nFinal SEM Structure (v4):")
print("""
  Measurement Model:
    leisure_part =~ hill_q1 + mean_leisure_duration
  
  Structural Model:
    STA ~ socio + transport
    travel_time ~ STA + socio + transport
    leisure_part ~ STA + travel_time + socio + transport
  
  Where:
    socio = female, edu_high, poverty_rate, hh_6
    transport = mode_pt, pt_sub, active_mode (EXOGENOUS)
  
  Note: No Individual → Transport Mode paths (justified by Test 8 + practical considerations)
""")

SUMMARY: DAG Path Justification for SEM v4

Test 1: Latent Variable Justification
  - Diversity and duration are correlated and load on single factor
  - Latent 'leisure_participation' is justified

Test 2-5: Path Necessity Tests
  - Use ANOVA nested model comparisons
  - p < 0.05 → path is needed
  - p >= 0.05 → path may be omitted for parsimony

Test 6: Mediation Test
  - Sobel test for indirect effect via travel time
  - Significant mediation supports the hypothesized mechanism

Test 7: Covariate Selection
  - Only include significant household type dummies

Test 8: Individual → Transport Mode
  - Tests whether sociodemographics predict mode choice
  - Even if significant, paths are OMITTED because:
    a) Including them caused model non-convergence
    b) Mode choice is complex, better modeled separately
    c) Transport mode treated as exogenous in our framework


Final SEM Structure (v4):

  Measurement Model:
    leisure_part =~ hill_q1 + mean_leisure_duration

  Structural Mode